In [60]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

### Prepare Lazada's reviews dataset

DO NOT RUN THIS SCRIPT AGAIN!!!

In [2]:
# Testing with Lazada Indonesian Reviews
lazada_reviews_path = '../data/lazada_reviews/lazada-reviews-without-sentiment.csv'

# Remove unused columns and remove missing values
lazada_df = pd.read_csv(lazada_reviews_path, header=None, low_memory=False)
lazada_df.columns = lazada_df.iloc[0]
lazada_df = lazada_df[1:]
lazada_df = lazada_df.reset_index(drop=True)
lazada_df = lazada_df[['reviewContent']].rename(columns={'reviewContent': 'text'})
lazada_df = lazada_df.dropna()
lazada_df['sentiment'] = 'isi ini ya ges (positive/neutral/negative)'

sample_df = lazada_df.sample(n=11000, random_state=42).reset_index(drop=True)
sample_df.to_csv('../data/lazada_reviews/train-lazada.csv')

In [3]:
remaining_df = lazada_df[~lazada_df['text'].isin(sample_df['text'])]

test_df = remaining_df.sample(n=500, random_state=99).reset_index(drop=True)
test_df.to_csv('../data/lazada_reviews/test-lazada.csv')

In [4]:
used_texts = set(sample_df['text']) | set(test_df['text'])

new_remaining_df = lazada_df[~lazada_df['text'].isin(used_texts)]

valid_df = remaining_df.sample(n=1260, random_state=123).reset_index(drop=True)
valid_df.to_csv('../data/lazada_reviews/valid-lazada.csv')

valid_df

,text,sentiment
0,"Barang sesuai deskripsi, 2hri nyampe",isi ini ya ges (positive/neutral/negative)
1,barang sesuai pesenan. original,isi ini ya ges (positive/neutral/negative)
2,"Barang ny udah smp sesuai pesanan, packing ny ...",isi ini ya ges (positive/neutral/negative)
3,"Works, dan hadiah sesuai deskripsi. barang dat...",isi ini ya ges (positive/neutral/negative)
4,peking sangat aman barang ok senang slalu blan...,isi ini ya ges (positive/neutral/negative)
...,...,...
1255,"👍👍👍👍 sesuai pesanan, tapi lom di coba",isi ini ya ges (positive/neutral/negative)
1256,good..,isi ini ya ges (positive/neutral/negative)
1257,"Barang sesuai dengan deskripsi, pokoknya manta...",isi ini ya ges (positive/neutral/negative)
1258,alhamdulillah ....puas.wlau pengirimannya hm...,isi ini ya ges (positive/neutral/negative)


### Combine all data and resplitting for final model

In [19]:
neutral_manual_path = '../data/lazada_reviews/neutral-manual.csv'
prdect_path = '../data/PRDECT-ID/PRDECT-ID-dataset.csv'
smsa_train_dataset = '../data/smsa/train_preprocess.tsv'
smsa_valid_dataset = '../data/smsa/valid_preprocess.tsv'
smsa_test_dataset= '../data/smsa/test_preprocess.tsv'

In [32]:
neutral_manual_df = pd.read_csv(neutral_manual_path, header=None)
prdect_df = pd.read_csv(prdect_path, header=None)
smsa_train_df = pd.read_csv(smsa_train_dataset, sep='\t', header=None)
smsa_valid_df = pd.read_csv(smsa_valid_dataset, sep='\t', header=None)
smsa_test_df = pd.read_csv(smsa_test_dataset, sep='\t', header=None)

In [42]:
# Prepare PRDECT-ID dataset
# Remove unused columns on PRDECT dataset
prdect_df = prdect_df.drop(columns=[0, 1, 2, 3, 4, 5, 6, 7, 10])
prdect_df = prdect_df[1:]
prdect_df = prdect_df.reset_index(drop=True)
prdect_df.columns = ['text', 'sentiment']
prdect_df['sentiment'] = prdect_df['sentiment'].apply(lambda x: 'positive' if x == 'Positive' else 'negative')


prdect_df

,text,sentiment
0,Alhamdulillah berfungsi dengan baik. Packaging...,positive
1,"barang bagus dan respon cepat, harga bersaing ...",positive
2,"barang bagus, berfungsi dengan baik, seler ram...",positive
3,bagus sesuai harapan penjual nya juga ramah. t...,positive
4,"Barang Bagus, pengemasan Aman, dapat Berfungsi...",positive
...,...,...
5395,"Harga bersaing, barang sesuai pesanan. Saya na...",positive
5396,Beli ini krn Anak & Istri mau liburan di Jakar...,positive
5397,"pengemasan barang baik, kondisi barang jg utuh...",positive
5398,Mungil tapi bekerja dng baik. Dan murahh terja...,positive


In [33]:
# Prepare the neutral manual dataset
neutral_manual_df.columns = ['text', 'sentiment']
neutral_manual_df

,text,sentiment
0,Barang sudah sampai namun belum bisa dibuka bl...,neutral
1,Kalau mau cicilan gmna cra nya??,neutral
2,"Bagi Yang Minat Order/Pesan Barang Kami,Silahk...",neutral
3,"hai sis,,,sya mau nyicil dan sya udh cek/lihat...",neutral
4,untuk pemasangan di tembok apa ad pihak dari l...,neutral
...,...,...
1496,Mantap respon baik ramah fast dan sangat memba...,neutral
1497,"garansi distributor, 3 tahun",neutral
1498,"Karena saya hanya bantu untuk membelikan, buka...",neutral
1499,barang sudah datang tepat waktu. rapi aman,neutral


In [40]:
smsa_train_df.columns = ['text', 'sentiment']
smsa_valid_df.columns = ['text', 'sentiment']
smsa_test_df.columns = ['text', 'sentiment']

,text,sentiment
0,kemarin gue datang ke tempat makan baru yang a...,negative
1,kayak nya sih gue tidak akan mau balik lagi ke...,negative
2,"kalau dipikir-pikir , sebenarnya tidak ada yan...",negative
3,ini pertama kalinya gua ke bank buat ngurusin ...,negative
4,waktu sampai dengan gue pernah disuruh ibu lat...,negative
...,...,...
495,kata nya tidur yang baik itu minimal enam jam ...,neutral
496,indonesia itu ada di benua asia .,neutral
497,salah satu kegemaran anak remaja indonesia sek...,neutral
498,melihat warna hijau bisa bikin mata jadi lebih...,positive


In [47]:
smsa_test_df

,text,sentiment
0,kemarin gue datang ke tempat makan baru yang a...,negative
1,kayak nya sih gue tidak akan mau balik lagi ke...,negative
2,"kalau dipikir-pikir , sebenarnya tidak ada yan...",negative
3,ini pertama kalinya gua ke bank buat ngurusin ...,negative
4,waktu sampai dengan gue pernah disuruh ibu lat...,negative
...,...,...
495,kata nya tidur yang baik itu minimal enam jam ...,neutral
496,indonesia itu ada di benua asia .,neutral
497,salah satu kegemaran anak remaja indonesia sek...,neutral
498,melihat warna hijau bisa bikin mata jadi lebih...,positive


In [51]:
# Combine all datasets into a single DataFrame
combined_df = pd.concat([neutral_manual_df, prdect_df, smsa_train_df, smsa_valid_df, smsa_test_df], ignore_index=True)
print(combined_df['sentiment'].value_counts())
print("Total samples:", len(combined_df))

combined_df

sentiment
positive    9938
negative    6855
neutral     2868
Name: count, dtype: int64
Total samples: 19661


,text,sentiment
0,Barang sudah sampai namun belum bisa dibuka bl...,neutral
1,Kalau mau cicilan gmna cra nya??,neutral
2,"Bagi Yang Minat Order/Pesan Barang Kami,Silahk...",neutral
3,"hai sis,,,sya mau nyicil dan sya udh cek/lihat...",neutral
4,untuk pemasangan di tembok apa ad pihak dari l...,neutral
...,...,...
19656,kata nya tidur yang baik itu minimal enam jam ...,neutral
19657,indonesia itu ada di benua asia .,neutral
19658,salah satu kegemaran anak remaja indonesia sek...,neutral
19659,melihat warna hijau bisa bikin mata jadi lebih...,positive


In [59]:
# Stratified split: train 80%, valid 10%, test 10%
train_df, temp_df = train_test_split(
    combined_df, test_size=0.2, stratify=combined_df['sentiment'], random_state=42
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['sentiment'], random_state=42
)

print('Train:', train_df['sentiment'].value_counts())
print('Train total count:', len(train_df), '\n')
print('Valid:', valid_df['sentiment'].value_counts())
print('Valid total count:', len(valid_df), '\n')
print('Test:', test_df['sentiment'].value_counts())
print('Test total count:', len(test_df))

train_df.to_csv('../data/combined/train_combined.csv', index=False)
valid_df.to_csv('../data/combined/valid_combined.csv', index=False)
test_df.to_csv('../data/combined/test_combined.csv', index=False)

Train: sentiment
positive    7950
negative    5484
neutral     2294
Name: count, dtype: int64
Train total count: 15728 

Valid: sentiment
positive    994
negative    685
neutral     287
Name: count, dtype: int64
Valid total count: 1966 

Test: sentiment
positive    994
negative    686
neutral     287
Name: count, dtype: int64 

Test total count: 1967


In [62]:
# Oversampling the minority class in the training set
positive_df = train_df[train_df['sentiment'] == 'positive']
negative_df = train_df[train_df['sentiment'] == 'negative']
neutral_df = train_df[train_df['sentiment'] == 'neutral']

max_count = max(len(positive_df), len(negative_df), len(neutral_df))

positive_upsampled = resample(positive_df,
    replace=True,
    n_samples=max_count,
    random_state=42
    )

negative_upsampled = resample(negative_df,
    replace=True,
    n_samples=max_count,
    random_state=42
    )

neutral_upsampled = resample(neutral_df,
    replace=True,
    n_samples=max_count,
    random_state=42
    )

train_upsampled_df = pd.concat([positive_upsampled, negative_upsampled, neutral_upsampled], ignore_index=True)

print('Upsampled Train:', train_upsampled_df['sentiment'].value_counts())
print('Upsampled Train total count:', len(train_upsampled_df))

train_upsampled_df.to_csv('../data/combined/train_combined_upsampled.csv', index=False)

Upsampled Train: sentiment
positive    7950
negative    7950
neutral     7950
Name: count, dtype: int64
Upsampled Train total count: 23850


In [ ]:
test_df

,text,sentiment
3383,pesan 50pc dikirim 25pc padahal bayar 50pc?? s...,negative
477,Bagaimana cara membatalkan pesanan barang ters...,neutral
19225,takdir politik ahy belum bisa ikut kontestasi ...,negative
6792,Mantap sesuai.,positive
3852,Produk sesuai deskripsi. Good communication. F...,positive
...,...,...
3400,"padahal udh atc bubble tapi gaada , paraaaaah ...",negative
9423,steak impor murah dan enak . menjadi salah sat...,positive
8129,kecil-kecil cabai rawit diusung pdip jadi cagu...,neutral
2505,produk sesuai deskripsi dan pengirimannya cepat,positive


In [67]:
# Create test set with unmasked labels
test_unmasked_df = test_df.copy()
test_unmasked_df['sentiment'] = test_unmasked_df['sentiment'].apply(lambda x: 'neutral')
test_unmasked_df.to_csv('../data/combined/test_combined_unmasked.csv', index=False)

test_unmasked_df

,text,sentiment
3383,pesan 50pc dikirim 25pc padahal bayar 50pc?? s...,neutral
477,Bagaimana cara membatalkan pesanan barang ters...,neutral
19225,takdir politik ahy belum bisa ikut kontestasi ...,neutral
6792,Mantap sesuai.,neutral
3852,Produk sesuai deskripsi. Good communication. F...,neutral
...,...,...
3400,"padahal udh atc bubble tapi gaada , paraaaaah ...",neutral
9423,steak impor murah dan enak . menjadi salah sat...,neutral
8129,kecil-kecil cabai rawit diusung pdip jadi cagu...,neutral
2505,produk sesuai deskripsi dan pengirimannya cepat,neutral
